In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

ACCENT = "#2F7A5C"
AMBER = "#9C7325"
RANDOM_STATE = 42

In [2]:
load_dotenv(override=True)

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.environ.get("DB_USER"),
    password=os.environ.get("DB_PASS"),
    host=os.environ.get("DB_HOST", "localhost"),
    port=int(os.environ.get("DB_PORT", "5432")),
    database=os.environ.get("DB_NAME"),
)
engine = create_engine(connection_url)

In [ ]:
#Load data from PostgreSQL

query = """
SELECT *
FROM metrics.scorecard_features;
"""

df = pd.read_sql_query(query, engine)

# Standardise column names
df.columns = df.columns.str.lower()

print("Dataset loaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()

Dataset loaded successfully!
Rows: 307,511
Columns: 44


,sk_id_curr,target,name_contract_type,name_income_type,name_education_type,name_family_status,name_housing_type,occupation_type,organization_type,flag_own_car,...,credit_goods_ratio,prior_loans_approved,prior_recency_days,prior_total_credit,prior_instalments_paid,prior_late_share,prior_max_dpd,bureau_n_credits,bureau_n_active,bureau_max_overdue_days
0,100002,1.00,Cash loans,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers,Business Entity Type 3,N,...,1.16,1,606.00,"179,055.00",19.00,0.00,-12.00,8.00,2.00,0.00
1,100003,0.00,Cash loans,State servant,Higher education,Married,House / apartment,Core staff,School,N,...,1.15,3,746.00,"1,452,573.00",25.00,0.00,-1.00,4.00,1.00,0.00
2,100004,0.00,Revolving loans,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers,Government,Y,...,1.00,1,815.00,"20,106.00",3.00,0.00,-3.00,2.00,0.00,0.00
3,100006,0.00,Cash loans,Working,Secondary / secondary special,Civil marriage,House / apartment,Laborers,Business Entity Type 3,N,...,1.05,5,181.00,"1,718,644.50",16.00,0.00,-1.00,NaN,NaN,NaN
4,100007,0.00,Cash loans,Working,Secondary / secondary special,Single / not married,House / apartment,Core staff,Religion,N,...,1.00,6,374.00,"999,832.50",66.00,0.24,12.00,1.00,0.00,0.00


In [5]:
# Check the data structure
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Data columns (total 44 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   sk_id_curr                   307511 non-null  int64  
 1   target                       307511 non-null  float64
 2   name_contract_type           307511 non-null  str    
 3   name_income_type             307511 non-null  str    
 4   name_education_type          307511 non-null  str    
 5   name_family_status           307511 non-null  str    
 6   name_housing_type            307511 non-null  str    
 7   occupation_type              211120 non-null  str    
 8   organization_type            307511 non-null  str    
 9   flag_own_car                 307511 non-null  str    
 10  flag_own_realty              307511 non-null  str    
 11  ext_source_1                 134133 non-null  float64
 12  ext_source_2                 306851 non-null  float64
 13  ext_source

In [6]:
# Target distribution
print(df["target"].value_counts())

print("\nTarget proportions:")
print(df["target"].value_counts(normalize=True).mul(100).round(2))

target
0.00    282686
1.00     24825
Name: count, dtype: int64

Target proportions:
target
0.00   91.93
1.00    8.07
Name: proportion, dtype: float64


In [7]:
# Missing values
missing_summary = (
    df.isnull()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(2)

missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_percentage
own_car_age,202929,65.99
ext_source_1,173378,56.38
occupation_type,96391,31.35
ext_source_3,60965,19.83
employment_years,55374,18.01
bureau_max_overdue_days,44020,14.31
bureau_n_active,44020,14.31
bureau_n_credits,44020,14.31
amt_req_credit_bureau_year,41519,13.50
prior_recency_days,17446,5.67



- Car age --> Missing primarily because applicants do not own cars -->	Retain ownership indicator; assess car age separately

- Previous loans --> Zero previous loans is a meaningful category -->	Preserve zero separately from missing

- Bureau information --> Three variables share exactly the same 44,020 missing records -->	Treat availability consistently

### Classify Predictors Automatically

In [10]:
# Exclude identifier and target from predictors
excluded_columns = ["sk_id_curr", "target"]

predictors = df.drop(columns=excluded_columns)

# Classify based on pandas data types
categorical_features = predictors.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_features = predictors.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))

print("\nCategorical variables:")
print(categorical_features)

print("\nNumerical variables:")
print(numerical_features)

Categorical features: 9
Numerical features: 33

Categorical variables:
['name_contract_type', 'name_income_type', 'name_education_type', 'name_family_status', 'name_housing_type', 'occupation_type', 'organization_type', 'flag_own_car', 'flag_own_realty']

Numerical variables:
['ext_source_1', 'ext_source_2', 'ext_source_3', 'amt_income_total', 'amt_credit', 'amt_annuity', 'amt_goods_price', 'age_years', 'employment_years', 'years_since_registration', 'years_since_id_change', 'years_since_phone_change', 'own_car_age', 'cnt_children', 'cnt_fam_members', 'region_population_relative', 'region_rating_client_w_city', 'obs_30_cnt_social_circle', 'def_30_cnt_social_circle', 'amt_req_credit_bureau_year', 'credit_income_ratio', 'annuity_income_ratio', 'credit_term_months', 'credit_goods_ratio', 'prior_loans_approved', 'prior_recency_days', 'prior_total_credit', 'prior_instalments_paid', 'prior_late_share', 'prior_max_dpd', 'bureau_n_credits', 'bureau_n_active', 'bureau_max_overdue_days']


C:\Users\MoteTheAnalyst\AppData\Local\Temp\ipykernel_12248\3046025334.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = predictors.select_dtypes(


In [14]:
# Examine numerical variables with low cardinality
discrete_summary = pd.DataFrame({
    "variable": numerical_features,
    "unique_values": [
        df[col].nunique(dropna=True)
        for col in numerical_features
    ],
    "sample_values": [
        sorted(df[col].dropna().unique().tolist())[:10]
        for col in numerical_features
    ]
})

discrete_summary = discrete_summary.sort_values(
    "unique_values"
)

display(discrete_summary)

,variable,unique_values,sample_values
16,region_rating_client_w_city,3,"[1.0, 2.0, 3.0]"
18,def_30_cnt_social_circle,10,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ..."
13,cnt_children,15,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ..."
14,cnt_fam_members,17,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ..."
31,bureau_n_active,23,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ..."
19,amt_req_credit_bureau_year,25,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ..."
24,prior_loans_approved,26,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]"
17,obs_30_cnt_social_circle,33,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ..."
30,bureau_n_credits,60,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ..."
12,own_car_age,62,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ..."


In [15]:
# Inpect high cardinality variables
uniqueness = pd.DataFrame({
    "variable": predictors.columns,
    "unique_values": [
        predictors[col].nunique(dropna=True)
        for col in predictors.columns
    ],
    "unique_ratio": [
        predictors[col].nunique(dropna=True) / len(df)
        for col in predictors.columns
    ]
})

display(
    uniqueness.sort_values(
        "unique_ratio",
        ascending=False
    ).head(15)
)

,variable,unique_values,unique_ratio
35,prior_total_credit,173011,0.56
10,ext_source_2,119831,0.39
9,ext_source_1,114584,0.37
30,annuity_income_ratio,88811,0.29
29,credit_income_ratio,48703,0.16
31,credit_term_months,39539,0.13
14,amt_annuity,13672,0.04
13,amt_credit,5603,0.02
37,prior_late_share,4967,0.02
18,years_since_registration,4713,0.02


In [12]:
# Inspect categrical variables
categorical_summary = pd.DataFrame({
    "variable": predictors.columns,
    "data_type": predictors.dtypes.astype(str).values,
    "unique_values": predictors.nunique(dropna=True).values,
    "missing_values": predictors.isnull().sum().values
})

categorical_summary = categorical_summary.sort_values(
    "unique_values"
)

categorical_summary

,variable,data_type,unique_values,missing_values
0,name_contract_type,str,2,0
7,flag_own_car,str,2,0
8,flag_own_realty,str,2,0
25,region_rating_client_w_city,float64,3,0
2,name_education_type,str,5,0
4,name_housing_type,str,6,0
3,name_family_status,str,6,0
1,name_income_type,str,8,0
27,def_30_cnt_social_circle,float64,10,1021
22,cnt_children,float64,15,0


Some have many possible values, so they will be grouped into sensible risk bands rather than create a separate bin for every value.

**Continuous numerical variables**

The remaining financial ratios, income amounts, credit amounts, external scores, age and time-related variables will initially be treated as continuous.

In [16]:

# Actual Values of descrete variables
discrete_candidates = [
    "region_rating_client_w_city",
    "cnt_children",
    "cnt_fam_members",
    "def_30_cnt_social_circle",
    "obs_30_cnt_social_circle",
    "amt_req_credit_bureau_year",
    "prior_loans_approved",
    "prior_instalments_paid",
    "bureau_n_credits",
    "bureau_n_active"
]

for col in discrete_candidates:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).sort_index().head(30))


--- region_rating_client_w_city ---
region_rating_client_w_city
1.00     34167
2.00    229484
3.00     43860
Name: count, dtype: int64

--- cnt_children ---
cnt_children
0.00     215371
1.00      61119
2.00      26749
3.00       3717
4.00        429
5.00         84
6.00         21
7.00          7
8.00          2
9.00          2
10.00         2
11.00         1
12.00         2
14.00         3
19.00         2
Name: count, dtype: int64

--- cnt_fam_members ---
cnt_fam_members
1.00      67847
2.00     158357
3.00      52601
4.00      24697
5.00       3478
6.00        408
7.00         81
8.00         20
9.00          6
10.00         3
11.00         1
12.00         2
13.00         1
14.00         2
15.00         1
16.00         2
20.00         2
NaN           2
Name: count, dtype: int64

--- def_30_cnt_social_circle ---
def_30_cnt_social_circle
0.00     271324
1.00      28328
2.00       5323
3.00       1192
4.00        253
5.00         56
6.00         11
7.00          1
8.00          1
34.00

## Stratified Train/Split

- 80% training data: used to learn bin boundaries, calculate WoE/IV and fit the logistic regression model.
- 20% test data: held back for final model evaluation.
- Stratification ensures both datasets retain approximately the original 8.07% default rate.

In [18]:
from sklearn.model_selection import train_test_split

# Keep applicant IDs separate for traceability
applicant_ids = df["sk_id_curr"]

# Separate predictors and target
X = df.drop(columns=["sk_id_curr", "target"])
y = df["target"]

# Stratified 80:20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Check the resulting datasets
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining default rate:")
print(f"{y_train.mean():.2%}")

print("\nTesting default rate:")
print(f"{y_test.mean():.2%}")

Training set: (246008, 42)
Testing set: (61503, 42)

Training default rate:
8.07%

Testing default rate:
8.07%


The matching default rates confirm that stratification worked. Both datasets retain the original class distribution, which is important for evaluating a credit-risk model fairly. There are 42 predictors available for modelling and the split preserves the original default rate.

In [19]:
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

print("\nTraining missing values:", X_train.isna().sum().sum())
print("Testing missing values:", X_test.isna().sum().sum())

Training observations: 246008
Testing observations: 61503
Training target distribution:
target
0.00   0.92
1.00   0.08
Name: proportion, dtype: float64

Testing target distribution:
target
0.00   0.92
1.00   0.08
Name: proportion, dtype: float64

Training missing values: 678548
Testing missing values: 169885


## The WoE/IV analysis

In [20]:
# Prepare the variables for binning
# Categorical variables
categorical_features = [
    "name_contract_type",
    "name_income_type",
    "name_education_type",
    "name_family_status",
    "name_housing_type",
    "occupation_type",
    "organization_type",
    "flag_own_car",
    "flag_own_realty"
]

# Ordinal variable
ordinal_features = [
    "region_rating_client_w_city"
]

# Numerical variables
numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features + ordinal_features
]

print("Categorical variables:", len(categorical_features))
print("Ordinal variables:", len(ordinal_features))
print("Numerical variables:", len(numerical_features))

Categorical variables: 9
Ordinal variables: 1
Numerical variables: 32


In [21]:
# Examine Categorical Distributions in the training set
for col in categorical_features:
    print(f"\n--- {col} ---")
    print(
        X_train[col]
        .value_counts(dropna=False)
        .to_string()
    )


--- name_contract_type ---
name_contract_type
Cash loans         222560
Revolving loans     23448

--- name_income_type ---
name_income_type
Working                 127043
Commercial associate     57273
Pensioner                44134
State servant            17518
Unemployed                  17
Student                     11
Businessman                  9
Maternity leave              3

--- name_education_type ---
name_education_type
Secondary / secondary special    174768
Higher education                  59802
Incomplete higher                  8289
Lower secondary                    3025
Academic degree                     124

--- name_family_status ---
name_family_status
Married                 157153
Single / not married     36359
Civil marriage           23812
Separated                15840
Widow                    12842
Unknown                      2

--- name_housing_type ---
name_housing_type
House / apartment      218383
With parents            11857
Municipal apartment    

### How categorical variables will be handled

- name_contract_type -->	Keep both categories

- name_income_type -->	Group very rare categories

- name_education_type -->	Keep categories initially; monitor Academic degree

- name_family_status -->	Group the two Unknown observations with an appropriate missing/other category

- name_housing_type -->	Keep all six initially

- occupation_type	--> Retain missing as its own category; monitor rare occupations

- organization_type -->	Group rare categories into broader groups if necessary

- flag_own_car -->	Keep both categories

- flag_own_realty -->	Keep both categories

**Important decisions**

Income type

There are four extremely small categories:
- Unemployed: 17
- Student: 11
- Businessman: 9
- Maternity leave: 3

**Occupation type**

There are 76,940 missing observations (about 31.3% of the training set). Keep missing as a separate category. This may carry useful information about the applicant's profile.

Some occupations have fewer than 500 observations. They can be grouped into "Other" if their individual bins prove too small.

**Organization type**

There are 58 categories, including XNA (44,143 observations). Do not automatically merge XNA with missing values or Other. First, establish whether it represents an unavailable organization type.

Two categories can have small populations but substantially different default risks. Next step: examine their default rates first.

In [22]:
# Profile default rate by category
cat_profile = X_train[categorical_features].copy()
cat_profile["target"] = y_train

for col in categorical_features:
    print(f"\n--- {col} ---")

    profile = (
        cat_profile.groupby(col, dropna=False)["target"]
        .agg(
            applicants="count",
            defaults="sum",
            default_rate="mean"
        )
        .sort_values("applicants", ascending=False)
    )

    profile["default_rate"] = (
        profile["default_rate"] * 100
    ).round(2)

    print(profile.to_string())


--- name_contract_type ---
                    applicants  defaults  default_rate
name_contract_type                                    
Cash loans              222560 18,572.00          8.34
Revolving loans          23448  1,288.00          5.49

--- name_income_type ---
                      applicants  defaults  default_rate
name_income_type                                        
Working                   127043 12,118.00          9.54
Commercial associate       57273  4,348.00          7.59
Pensioner                  44134  2,365.00          5.36
State servant              17518  1,024.00          5.85
Unemployed                    17      5.00         29.41
Student                       11      0.00          0.00
Businessman                    9      0.00          0.00
Maternity leave                3      0.00          0.00

--- name_education_type ---
                               applicants  defaults  default_rate
name_education_type                                          

## Decisions based on the observed default rates

**Variables to retain without grouping**

- name_contract_type: Cash loans have an 8.34% default rate versus 5.49% for revolving loans.

- name_education_type: The categories show useful differences.  but monitor Academic degree because it has only two defaults.

- name_housing_type: Keep all six initially. Rented apartments and living with parents have relatively high default rates.

- flag_own_car and flag_own_realty: Keep both categories.

- occupation_type: Retain the missing category separately. The differences between occupations are informative.

**Variables requiring consolidation**

*name_income_type*

Group the four rare categories into Other:
- Unemployed
- Student
- Businessman
- Maternity leave

Their individual default rates are too unreliable for separate WoE estimates.

*name_family_status*

Merge Unknown into Other or a missing/unknown category. With only two applicants, it cannot support a reliable standalone estimate.

*organization_type*

This variable needs more care. For example:
- Transport: type 3: 15.77% default rate.
- Construction: 11.95%.
- XNA: 5.37%.
- Business Entity Type 3: 9.27%.

These differences suggest that indiscriminately merging categories could destroy useful risk information.



### Apply the initial category groupings

In [23]:
# Create working copies
X_train_woe = X_train.copy()
X_test_woe = X_test.copy()

# 1. Group rare income categories
income_mapping = {
    "Unemployed": "Other",
    "Student": "Other",
    "Businessman": "Other",
    "Maternity leave": "Other"
}

# 2. Group unknown family status
family_mapping = {
    "Unknown": "Other/unknown"
}

# 3. Combine Academic degree with Higher education
education_mapping = {
    "Academic degree": "Higher education"
}

# 4. Group rare organization categories
rare_organizations = [
    "Trade: type 6",
    "Insurance",
    "Industry: type 5",
    "Emergency",
    "Telecom",
    "Industry: type 2",
    "Advertising",
    "Realtor",
    "Culture",
    "Industry: type 12",
    "Trade: type 1",
    "Legal Services",
    "Mobile",
    "Cleaning",
    "Transport: type 1",
    "Industry: type 6",
    "Industry: type 10",
    "Religion",
    "Trade: type 4",
    "Industry: type 13",
    "Trade: type 5",
    "Industry: type 8"
]

organization_mapping = {
    category: "Rare organization"
    for category in rare_organizations
}

# Apply identical mappings to training and testing data
for dataset in [X_train_woe, X_test_woe]:

    dataset["name_income_type"] = (
        dataset["name_income_type"].replace(income_mapping)
    )

    dataset["name_family_status"] = (
        dataset["name_family_status"].replace(family_mapping)
    )

    dataset["name_education_type"] = (
        dataset["name_education_type"].replace(education_mapping)
    )

    dataset["organization_type"] = (
        dataset["organization_type"].replace(organization_mapping)
    )

# Represent missing categorical values explicitly
for col in categorical_features:
    X_train_woe[col] = X_train_woe[col].fillna("Missing")
    X_test_woe[col] = X_test_woe[col].fillna("Missing")

In [24]:
# Verify the transformations
for col in categorical_features:
    print(f"\n--- {col} ---")
    print(X_train_woe[col].value_counts().to_string())


--- name_contract_type ---
name_contract_type
Cash loans         222560
Revolving loans     23448

--- name_income_type ---
name_income_type
Working                 127043
Commercial associate     57273
Pensioner                44134
State servant            17518
Other                       40

--- name_education_type ---
name_education_type
Secondary / secondary special    174768
Higher education                  59926
Incomplete higher                  8289
Lower secondary                    3025

--- name_family_status ---
name_family_status
Married                 157153
Single / not married     36359
Civil marriage           23812
Separated                15840
Widow                    12842
Other/unknown                2

--- name_housing_type ---
name_housing_type
House / apartment      218383
With parents            11857
Municipal apartment      8916
Rented apartment         3873
Office apartment         2097
Co-op apartment           882

--- occupation_type ---
occupation_

### calculate WoE and IV for categorical variables

Using the training set only.

The formulas are:


### Weight of Evidence (WoE)

$$
WoE_i = \ln\left(
\frac{\% \text{Non-defaults}_i}
{\% \text{Defaults}_i}
\right)
$$

### Information Value (IV)

$$
IV = \sum_i
\left(
\% \text{Non-defaults}_i -
\% \text{Defaults}_i
\right)
\times WoE_i
$$


We'll use a small smoothing adjustment to avoid infinite WoE values.

In [25]:
# Create the WoE calculation function
def calculate_woe_iv(data, target, feature, smoothing=0.5):

    temp = pd.DataFrame({
        "feature": data[feature].fillna("Missing"),
        "target": target
    })

    grouped = temp.groupby("feature", dropna=False)["target"].agg(
        applicants="count",
        defaults="sum"
    )

    grouped["non_defaults"] = (
        grouped["applicants"] - grouped["defaults"]
    )

    total_defaults = grouped["defaults"].sum()
    total_non_defaults = grouped["non_defaults"].sum()

    n_bins = len(grouped)

    grouped["default_distribution"] = (
        (grouped["defaults"] + smoothing) /
        (total_defaults + smoothing * n_bins)
    )

    grouped["non_default_distribution"] = (
        (grouped["non_defaults"] + smoothing) /
        (total_non_defaults + smoothing * n_bins)
    )

    grouped["woe"] = np.log(
        grouped["non_default_distribution"] /
        grouped["default_distribution"]
    )

    grouped["iv_component"] = (
        grouped["non_default_distribution"] -
        grouped["default_distribution"]
    ) * grouped["woe"]

    grouped["default_rate"] = (
        grouped["defaults"] / grouped["applicants"]
    )

    iv = grouped["iv_component"].sum()

    return grouped.sort_values("woe"), iv

In [44]:
# Calculate IV for all categorical variables

"""
IV	        Interpretation
Below 0.02	Little predictive power
0.02–0.10	Weak
0.10–0.30	Medium
0.30–0.50	Strong
Above 0.50	Suspiciously predictive; investigate

"""
categorical_iv_results = []

for col in categorical_features:

    bins, iv = calculate_woe_iv(
        X_train_woe,
        y_train,
        col
    )

    categorical_iv_results.append({
        "variable": col,
        "IV": iv,
        "number_of_bins": len(bins)
    })

categorical_iv = pd.DataFrame(
    categorical_iv_results
).sort_values("IV", ascending=False)

categorical_iv.round(4)

,variable,IV,number_of_bins
5,occupation_type,0.08,19
6,organization_type,0.07,37
1,name_income_type,0.06,5
2,name_education_type,0.05,4
3,name_family_status,0.02,6
4,name_housing_type,0.02,6
0,name_contract_type,0.01,2
7,flag_own_car,0.01,2
8,flag_own_realty,0.00,2


- Occupation type has the highest IV (0.08), suggesting that employment categories provide some information about repayment risk.

- Organization type and income type also contribute information about borrower risk.

- Education type has some predictive value, although its IV is relatively modest.

- Car and property ownership have little standalone discriminatory power in this dataset.

N/B: IV measures individual predictive power, not the contribution of a variable within a multivariable model. Some variables may provide complementary information to numerical predictors. 

In [28]:
# inspect the individual WoE bins
for col in categorical_features:

    bins, iv = calculate_woe_iv(
        X_train_woe,
        y_train,
        col
    )

    print(f"\n{'='*60}")
    print(f"Variable: {col} | IV: {iv:.4f}")
    print(f"{'='*60}")

    print(
        bins[
            [
                "applicants",
                "defaults",
                "default_rate",
                "woe",
                "iv_component"
            ]
        ].round(4).to_string()
    )


Variable: name_contract_type | IV: 0.0149
                 applicants  defaults  default_rate   woe  iv_component
feature                                                                
Cash loans           222560 18,572.00          0.08 -0.04          0.00
Revolving loans       23448  1,288.00          0.05  0.41          0.01

Variable: name_income_type | IV: 0.0559
                      applicants  defaults  default_rate   woe  iv_component
feature                                                                     
Other                         40      5.00          0.12 -0.57          0.00
Working                   127043 12,118.00          0.10 -0.18          0.02
Commercial associate       57273  4,348.00          0.08  0.07          0.00
State servant              17518  1,024.00          0.06  0.35          0.01
Pensioner                  44134  2,365.00          0.05  0.44          0.03

Variable: name_education_type | IV: 0.0489
                               applicants  de

- Income type: Pensioners have a WoE of 0.44, while Working applicants have -0.18. This suggests meaningful differences in repayment risk across income groups.

- Education: Higher education has a WoE of 0.43, compared with -0.35 for lower secondary education. This is a useful risk differentiation.

- Occupation: Low-skill Laborers have the highest observed default rate (18%) and a WoE of -0.92. Accountants have a 5% default rate and a WoE of 0.59.

- Organization: Transport: type 3 has a 16% default rate and WoE of -0.76. XNA has a 5% default rate and WoE of 0.44. These differences suggest organization type carries useful information.

- Ownership: Car ownership has limited predictive power, while property ownership contributes almost nothing individually.

### Numerical variable binning

The goal is bins that;

- Have sufficient observations.

- Separate applicants by default risk.

- Produce interpretable WoE values.

- Have reasonably monotonic risk patterns where appropriate.

- Handle missing values consistently.

In [32]:
# Identify the numerical variables with missing values
numerical_missing = (
    X_train[numerical_features]
    .isna()
    .sum()
    .to_frame("missing_count")
)

numerical_missing["missing_pct"] = (
    numerical_missing["missing_count"] /
    len(X_train) * 100
)

numerical_missing = numerical_missing[
    numerical_missing["missing_count"] > 0
].sort_values("missing_pct", ascending=False)

numerical_missing.round(2)


,missing_count,missing_pct
own_car_age,162418,66.02
ext_source_1,138595,56.34
ext_source_3,48805,19.84
employment_years,44143,17.94
bureau_n_active,35244,14.33
bureau_n_credits,35244,14.33
bureau_max_overdue_days,35244,14.33
amt_req_credit_bureau_year,33244,13.51
prior_recency_days,13985,5.68
prior_total_credit,13985,5.68


In [ ]:
missingness_checks = [
    "prior_loans_approved",
    "prior_recency_days",
    "prior_total_credit",
    "prior_instalments_paid",
    "prior_late_share",
    "prior_max_dpd",
    "bureau_n_credits",
    "bureau_n_active",
    "bureau_max_overdue_days"
]

for col in missingness_checks:
    print(f"{col}:")
    print(
        X_train.groupby(
            X_train[col].isna()
        )["prior_loans_approved"].agg(
            applicants="count",
            average_previous_loans="mean"
        )
    )
    print()

prior_loans_approved:
                      applicants  average_previous_loans
prior_loans_approved                                    
False                     246008                    2.88

prior_recency_days:
                    applicants  average_previous_loans
prior_recency_days                                    
False                   232023                    3.06
True                     13985                    0.00

prior_total_credit:
                    applicants  average_previous_loans
prior_total_credit                                    
False                   232023                    3.06
True                     13985                    0.00

prior_instalments_paid:
                        applicants  average_previous_loans
prior_instalments_paid                                    
False                       233317                    3.04
True                         12691                    0.03

prior_late_share:
                  applicants  average_previou

Decision: Preserve these missing values as a separate bin rather than replacing them with zero or the median. Zero previous loans and missing previous-loan information should not automatically be treated as equivalent.

In [34]:
# Create initial quantile bins using training data
ext_source_1_bins = pd.qcut(
    X_train["ext_source_1"],
    q=10,
    duplicates="drop"
)

# Preserve missing observations as a separate group
ext_source_1_bins = (
    ext_source_1_bins
    .astype("object")
    .fillna("Missing")
)

# Calculate WoE and IV
X_train_woe["ext_source_1_bin"] = ext_source_1_bins

ext_source_1_woe, ext_source_1_iv = calculate_woe_iv(
    X_train_woe,
    y_train,
    "ext_source_1_bin"
)

print(f"Information Value: {ext_source_1_iv:.4f}")

display(ext_source_1_woe)

Information Value: 0.1512


,applicants,defaults,non_defaults,default_distribution,non_default_distribution,woe,iv_component,default_rate
feature,,,,,,,,
"(0.013600000000000001, 0.212]",10742,"1,870.00","8,872.00",0.09,0.04,-0.88,0.05,0.17
"(0.212, 0.296]",10741,"1,250.00","9,491.00",0.06,0.04,-0.41,0.01,0.12
"(0.296, 0.368]",10741,981.00,"9,760.00",0.05,0.04,-0.14,0.00,0.09
Missing,138595,"11,815.00","126,780.00",0.59,0.56,-0.06,0.00,0.09
"(0.368, 0.437]",10741,907.00,"9,834.00",0.05,0.04,-0.05,0.00,0.08
"(0.437, 0.505]",10742,732.00,"10,010.00",0.04,0.04,0.18,0.00,0.07
"(0.505, 0.573]",10741,640.00,"10,101.00",0.03,0.04,0.33,0.00,0.06
"(0.573, 0.64]",10741,555.00,"10,186.00",0.03,0.05,0.48,0.01,0.05
"(0.64, 0.71]",10741,466.00,"10,275.00",0.02,0.05,0.66,0.01,0.04


Three observations:

1. Strong risk gradient: Default rates decline from approximately 17 % in the lowest bin to 3 % in the highest.

2. Monotonic relationship: Each successive numerical bin has a lower default rate. This is desirable for a traditional scorecard.

3. Missing values: The missing group has an 9% default rate, slightly above the overall training default rate of approximately 8.07%.

The missing group is large (56.34% of the training sample), so retaining it as a separate bin is important.

In [35]:
# verify the exact IV and risk ordering

print(f"ext_source_1 IV: {ext_source_1_iv:.4f}")

print(
    ext_source_1_woe[
        ["applicants", "defaults", "default_rate", "woe"]
    ].sort_values("default_rate", ascending=False)
)

ext_source_1 IV: 0.1512
                               applicants  defaults  default_rate   woe
feature                                                                
(0.013600000000000001, 0.212]       10742  1,870.00          0.17 -0.88
(0.212, 0.296]                      10741  1,250.00          0.12 -0.41
(0.296, 0.368]                      10741    981.00          0.09 -0.14
Missing                            138595 11,815.00          0.09 -0.06
(0.368, 0.437]                      10741    907.00          0.08 -0.05
(0.437, 0.505]                      10742    732.00          0.07  0.18
(0.505, 0.573]                      10741    640.00          0.06  0.33
(0.573, 0.64]                       10741    555.00          0.05  0.48
(0.64, 0.71]                        10741    466.00          0.04  0.66
(0.71, 0.787]                       10741    367.00          0.03  0.91
(0.787, 0.952]                      10742    277.00          0.03  1.20


In [36]:
# a reusable function for the remaining numerical variables

def numerical_woe_binning(data, target, feature, q=10, smoothing=0.5):

    # Create quantile bins
    bins = pd.qcut(
        data[feature],
        q=q,
        duplicates="drop"
    )

    # Preserve missing values
    bins = bins.astype("object").fillna("Missing")

    # Calculate WoE and IV
    temp = pd.DataFrame({
        "feature": bins,
        "target": target
    })

    grouped = temp.groupby("feature", observed=False)["target"].agg(
        applicants="count",
        defaults="sum"
    )

    grouped["non_defaults"] = (
        grouped["applicants"] - grouped["defaults"]
    )

    total_defaults = grouped["defaults"].sum()
    total_non_defaults = grouped["non_defaults"].sum()
    n_bins = len(grouped)

    grouped["default_distribution"] = (
        (grouped["defaults"] + smoothing) /
        (total_defaults + smoothing * n_bins)
    )

    grouped["non_default_distribution"] = (
        (grouped["non_defaults"] + smoothing) /
        (total_non_defaults + smoothing * n_bins)
    )

    grouped["woe"] = np.log(
        grouped["non_default_distribution"] /
        grouped["default_distribution"]
    )

    grouped["iv_component"] = (
        grouped["non_default_distribution"] -
        grouped["default_distribution"]
    ) * grouped["woe"]

    grouped["default_rate"] = (
        grouped["defaults"] / grouped["applicants"]
    )

    return grouped, grouped["iv_component"].sum(), bins

In [38]:
# Using the function on the remaining 32 variables
numerical_iv_results = []

numerical_woe_tables = {}
numerical_bin_labels = {}

for feature in numerical_features:

    woe_table, iv, bin_labels = numerical_woe_binning(
        X_train,
        y_train,
        feature,
        q=10
    )

    numerical_iv_results.append({
        "feature": feature,
        "iv": iv,
        "number_of_bins": len(woe_table)
    })

    numerical_woe_tables[feature] = woe_table
    numerical_bin_labels[feature] = bin_labels


numerical_iv_summary = (
    pd.DataFrame(numerical_iv_results)
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)

numerical_iv_summary.round(4)

,feature,iv,number_of_bins
0,ext_source_3,0.33,11
1,ext_source_2,0.30,11
2,ext_source_1,0.15,11
3,credit_term_months,0.14,11
4,employment_years,0.11,11
5,amt_goods_price,0.09,11
6,age_years,0.09,10
7,credit_goods_ratio,0.07,9
8,prior_late_share,0.06,7
9,bureau_n_active,0.05,6


- External credit scores dominate. All three external sources rank in the top three.

- Loan term matters. credit_term_months has an IV of 0.14, suggesting a useful relationship with default risk.

- Employment history matters. employment_years has an IV of 0.11.

- Previous repayment behaviour (prior_late_share) and bureau activity also show useful, though weaker, signals.

- Several variables have very low IV. This does not automatically mean they should be removed; some may add information when combined with other predictors.

In [39]:
# Inspect top 5 variables
top_features = numerical_iv_summary.head(5)["feature"].tolist()

for feature in top_features:
    print(f"\n{'='*60}")
    print(f"FEATURE: {feature}")
    print(f"IV: {numerical_iv_summary.loc[
        numerical_iv_summary['feature'] == feature, 'iv'
    ].iloc[0]:.4f}")

    display(
        numerical_woe_tables[feature]
        .sort_values("default_rate", ascending=False)
        [[
            "applicants",
            "defaults",
            "default_rate",
            "woe",
            "iv_component"
        ]]
    )


FEATURE: ext_source_3
IV: 0.3296


,applicants,defaults,default_rate,woe,iv_component
feature,,,,,
"(-0.000473, 0.228]",19723,"3,928.00",0.20,-1.04,0.13
"(0.228, 0.33]",19757,"2,523.00",0.13,-0.51,0.03
"(0.33, 0.408]",20001,"1,956.00",0.10,-0.21,0.00
Missing,48805,"4,528.00",0.09,-0.15,0.00
"(0.408, 0.476]",19842,"1,576.00",0.08,0.02,0.00
"(0.476, 0.535]",19513,"1,206.00",0.06,0.29,0.01
"(0.535, 0.59]",19506,"1,042.00",0.05,0.44,0.01
"(0.59, 0.643]",20444,990.00,0.05,0.55,0.02
"(0.643, 0.694]",19808,812.00,0.04,0.72,0.03



FEATURE: ext_source_2
IV: 0.3030


,applicants,defaults,default_rate,woe,iv_component
feature,,,,,
"(-0.0009999183, 0.215]",24548,"4,487.00",0.18,-0.93,0.13
"(0.215, 0.34]",24548,"2,970.00",0.12,-0.45,0.02
"(0.34, 0.44]",24547,"2,423.00",0.10,-0.22,0.01
"(0.44, 0.512]",24549,"2,065.00",0.08,-0.04,0.00
Missing,531,41.00,0.08,0.04,0.00
"(0.512, 0.566]",24551,"1,833.00",0.07,0.08,0.00
"(0.566, 0.608]",24543,"1,611.00",0.07,0.22,0.00
"(0.608, 0.646]",24548,"1,437.00",0.06,0.35,0.01
"(0.646, 0.682]",24550,"1,234.00",0.05,0.51,0.02



FEATURE: ext_source_1
IV: 0.1512


,applicants,defaults,default_rate,woe,iv_component
feature,,,,,
"(0.013600000000000001, 0.212]",10742,"1,870.00",0.17,-0.88,0.05
"(0.212, 0.296]",10741,"1,250.00",0.12,-0.41,0.01
"(0.296, 0.368]",10741,981.00,0.09,-0.14,0.00
Missing,138595,"11,815.00",0.09,-0.06,0.00
"(0.368, 0.437]",10741,907.00,0.08,-0.05,0.00
"(0.437, 0.505]",10742,732.00,0.07,0.18,0.00
"(0.505, 0.573]",10741,640.00,0.06,0.33,0.00
"(0.573, 0.64]",10741,555.00,0.05,0.48,0.01
"(0.64, 0.71]",10741,466.00,0.04,0.66,0.01



FEATURE: credit_term_months
IV: 0.1419


,applicants,defaults,default_rate,woe,iv_component
feature,,,,,
"(20.0, 22.038]",22134,"3,136.00",0.14,-0.63,0.05
"(10.528, 14.658]",24625,"2,803.00",0.11,-0.38,0.02
"(17.791, 19.537]",24674,"2,483.00",0.10,-0.24,0.01
"(14.658, 17.791]",24564,"2,411.00",0.10,-0.21,0.01
"(25.215, 30.143]",24685,"1,817.00",0.07,0.10,0.00
"(30.143, 34.063]",25155,"1,771.00",0.07,0.15,0.00
"(19.537, 20.0]",26989,"1,584.00",0.06,0.34,0.01
"(34.063, 45.305]",23898,"1,377.00",0.06,0.36,0.01
"(22.038, 25.215]",24661,"1,266.00",0.05,0.48,0.02



FEATURE: employment_years
IV: 0.1121


,applicants,defaults,default_rate,woe,iv_component
feature,,,,,
"(0.9, 1.7]",20786,"2,400.00",0.12,-0.40,0.02
"(-0.001, 0.9]",21104,"2,341.00",0.11,-0.35,0.01
"(1.7, 2.5]",19637,"2,159.00",0.11,-0.34,0.01
"(2.5, 3.4]",19843,"2,036.00",0.10,-0.26,0.01
"(3.4, 4.5]",20281,"1,908.00",0.09,-0.17,0.00
"(4.5, 5.9]",19961,"1,667.00",0.08,-0.04,0.00
"(5.9, 7.6]",19752,"1,497.00",0.08,0.07,0.00
"(7.6, 10.1]",20678,"1,437.00",0.07,0.16,0.00
"(10.1, 14.6]",19797,"1,151.00",0.06,0.35,0.01


The three external credit scores are behaving very well. credit_term_months needs some refinement, while employment_years has an interesting missing-value pattern.

In [40]:
# Investigate external-score correlation
external_scores = [
    "ext_source_1",
    "ext_source_2",
    "ext_source_3"
]

external_correlation = (
    X_train[external_scores]
    .corr(method="spearman")
)

external_correlation.round(3)

,ext_source_1,ext_source_2,ext_source_3
ext_source_1,1.00,0.22,0.18
ext_source_2,0.22,1.00,0.11
ext_source_3,0.18,0.11,1.00


- The three variables have relatively low pairwise correlations.

- They may capture different aspects of applicants' creditworthiness.

### Assess the ordinal variable

In [41]:
region_woe, region_iv = calculate_woe_iv(
    X_train,
    y_train,
    "region_rating_client_w_city"
)

print(f"Region rating IV: {region_iv:.4f}")

display(region_woe)

Region rating IV: 0.0516


,applicants,defaults,non_defaults,default_distribution,non_default_distribution,woe,iv_component,default_rate
feature,,,,,,,,
3.00,35063,"4,019.00","31,044.00",0.20,0.14,-0.39,0.03,0.11
2.00,183622,"14,511.00","169,111.00",0.73,0.75,0.02,0.00,0.08
1.00,27323,"1,330.00","25,993.00",0.07,0.11,0.54,0.03,0.05


- Rating 1 has the lowest default rate.

- Rating 2 is close to the overall default rate.

- Rating 3 has the highest default rate.

In [42]:
# calculate its exact IV
print(f"Region rating IV: {region_iv:.4f}")

Region rating IV: 0.0516


### Consolidate the IV rankings For a complete view of all 42 predictors

In [48]:
# Convert categorical IV results into a DataFrame
categorical_iv_summary = pd.DataFrame(categorical_iv_results)

# Prepare ordinal IV results
ordinal_iv_summary = pd.DataFrame([{
    "variable": "region_rating_client_w_city",
    "IV": region_iv,
    "number_of_bins": len(region_woe)
}])

# Standardize numerical IV column names
numerical_iv_summary = numerical_iv_summary.rename(
    columns={"feature": "variable", "iv": "IV"}
)

# Combine all three feature groups
all_iv_summary = pd.concat(
    [
        categorical_iv_summary,
        numerical_iv_summary,
        ordinal_iv_summary
    ],
    ignore_index=True
)

# Rank features by IV
all_iv_summary = (
    all_iv_summary
    .sort_values("IV", ascending=False)
    .reset_index(drop=True)
)

all_iv_summary.round(4)

,variable,IV,number_of_bins
0,ext_source_3,0.33,11
1,ext_source_2,0.30,11
2,ext_source_1,0.15,11
3,credit_term_months,0.14,11
4,employment_years,0.11,11
5,amt_goods_price,0.09,11
6,age_years,0.09,10
7,occupation_type,0.08,19
8,organization_type,0.07,37
9,credit_goods_ratio,0.07,9


### Predictor selection and Redundancy Analysis

In [49]:
# Numerical predictor redundancy analysis
# Include numerical and ordinal predictors
correlation_features = numerical_features + [
    "region_rating_client_w_city"
]

# Calculate Spearman correlation using training data only
spearman_corr = X_train[correlation_features].corr(
    method="spearman"
)

# Extract unique feature pairs
correlation_pairs = (
    spearman_corr
    .where(
        np.triu(
            np.ones(spearman_corr.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

correlation_pairs.columns = [
    "feature_1",
    "feature_2",
    "spearman_correlation"
]

# Retain potentially strong relationships
high_corr_pairs = (
    correlation_pairs[
        correlation_pairs["spearman_correlation"].abs() >= 0.70
    ]
    .assign(
        absolute_correlation=lambda x:
            x["spearman_correlation"].abs()
    )
    .sort_values(
        "absolute_correlation",
        ascending=False
    )
    .reset_index(drop=True)
)

high_corr_pairs.round(3)

,feature_1,feature_2,spearman_correlation,absolute_correlation
0,amt_credit,amt_goods_price,0.98,0.98
1,prior_late_share,prior_max_dpd,0.85,0.85
2,amt_credit,amt_annuity,0.83,0.83
3,amt_annuity,amt_goods_price,0.83,0.83
4,cnt_children,cnt_fam_members,0.81,0.81
5,credit_income_ratio,annuity_income_ratio,0.80,0.80
6,amt_credit,credit_income_ratio,0.75,0.75
7,prior_loans_approved,prior_total_credit,0.75,0.75
8,prior_loans_approved,prior_instalments_paid,0.74,0.74
9,amt_goods_price,credit_income_ratio,0.73,0.73


- The pairs are highly correlated (>0.70)
- High correlation does not automatically mean a variable must be removed. We'll use predictive strength, interpretability, missingness and model stability together.

In [50]:
# examine the strongest redundancy
# Compare predictive strength and missingness
redundancy_candidates = [
    "amt_credit",
    "amt_goods_price",
    "amt_annuity",
    "cnt_children",
    "cnt_fam_members",
    "prior_late_share",
    "prior_max_dpd",
    "credit_income_ratio",
    "annuity_income_ratio",
    "prior_loans_approved",
    "prior_total_credit",
    "prior_instalments_paid"
]

candidate_comparison = (
    all_iv_summary[
        all_iv_summary["variable"].isin(redundancy_candidates)
    ]
    .merge(
        X_train[redundancy_candidates].isna().mean()
        .mul(100)
        .rename("missing_pct"),
        left_on="variable",
        right_index=True
    )
    .sort_values("IV", ascending=False)
)

candidate_comparison.round(3)

,variable,IV,number_of_bins,missing_pct
5,amt_goods_price,0.09,11,0.09
10,prior_late_share,0.06,7,5.16
15,amt_credit,0.04,10,0.00
17,prior_max_dpd,0.04,11,5.16
20,prior_total_credit,0.03,11,5.68
22,amt_annuity,0.03,11,0.00
30,prior_instalments_paid,0.01,11,5.16
32,credit_income_ratio,0.01,10,0.00
33,prior_loans_approved,0.01,6,0.00
34,annuity_income_ratio,0.01,11,0.00


In [53]:
"""
VIF below 5: generally acceptable.
VIF 5–10: investigate.
VIF above 10: potentially serious multicollinearity.

"""



# Check multicollinearity using VIF
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Prepare numerical and ordinal predictors
vif_features = numerical_features + [
    "region_rating_client_w_city"
]

X_vif = X_train[vif_features].copy()

# Impute missing values using training-set medians
imputer = SimpleImputer(strategy="median")
X_vif_imputed = imputer.fit_transform(X_vif)

# Standardize predictors for numerical stability
scaler = StandardScaler()
X_vif_scaled = scaler.fit_transform(X_vif_imputed)

# Calculate VIF
vif_results = pd.DataFrame({
    "variable": vif_features,
    "VIF": [
        variance_inflation_factor(X_vif_scaled, i)
        for i in range(X_vif_scaled.shape[1])
    ]
})

vif_results = vif_results.sort_values(
    "VIF", ascending=False
).reset_index(drop=True)

vif_results.round(2)

,variable,VIF
0,amt_credit,126.17
1,amt_goods_price,116.81
2,credit_income_ratio,16.23
3,annuity_income_ratio,11.37
4,amt_annuity,9.80
5,credit_term_months,7.11
6,cnt_children,4.69
7,cnt_fam_members,4.57
8,credit_goods_ratio,3.07
9,prior_loans_approved,2.41


- The VIF results confirm that we have a multicollinearity problem concentrated around loan amounts and affordability ratios, rather than across the entire predictor set.

| Variable              | IV   | VIF    | Decision       |
|-----------------------|-----:|-------:|----------------|
| amt_goods_price       | 0.09 | 116.81 | Retain         |
| amt_credit            | 0.04 | 126.17 | Remove         |
| amt_annuity           | 0.03 | 9.80   | Retain for now |
| credit_income_ratio   | 0.01 | 16.23  | Remove         |
| annuity_income_ratio  | 0.01 | 11.37  | Remove         |
| credit_term_months    | 0.14 | 7.11   | Retain         |
| cnt_children          | 0.00 | 4.69   | Remove         |
| cnt_fam_members       | 0.00 | 4.57   | Remove         |

- amt_goods_price has a much higher IV than amt_credit, while their correlation is 0.98. Keeping the stronger candidate is a reasonable starting point.
- amt_annuity despite its VIF of 9.80 It represents the repayment obligation, which is conceptually different from the amount borrowed. We can assess whether it adds value after the initial model.

In [54]:
# Apply the provisional exclusion
# Provisional exclusions based on IV and redundancy analysis
initial_exclusions = [
    "amt_credit",
    "credit_income_ratio",
    "annuity_income_ratio",
    "cnt_children",
    "cnt_fam_members"
]

# Create the reduced predictor sets
selected_features = [
    col for col in X_train.columns
    if col not in initial_exclusions
]

X_train_selected = X_train[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

print("Original predictors:", X_train.shape[1])
print("Selected predictors:", len(selected_features))
print("Excluded predictors:", len(initial_exclusions))

print("\nExcluded variables:")
print(initial_exclusions)

Original predictors: 42
Selected predictors: 37
Excluded predictors: 5

Excluded variables:
['amt_credit', 'credit_income_ratio', 'annuity_income_ratio', 'cnt_children', 'cnt_fam_members']


### Categorical Redundancy using Cramer's V

In [55]:
"""
Cramér's V	Interpretation
Below 0.10	Very weak association
0.10–0.30	Weak association
0.30–0.50	Moderate association
Above 0.50	Strong association

"""


import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from itertools import combinations

categorical_features = [
    'name_contract_type',
    'name_income_type',
    'name_education_type',
    'name_family_status',
    'name_housing_type',
    'occupation_type',
    'organization_type',
    'flag_own_car',
    'flag_own_realty',
    'region_rating_client_w_city'
]

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)

    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape

    phi2 = chi2 / n

    # Bias correction
    phi2_corrected = max(
        0, phi2 - ((k - 1) * (r - 1)) / (n - 1)
    )

    r_corrected = r - ((r - 1) ** 2) / (n - 1)
    k_corrected = k - ((k - 1) ** 2) / (n - 1)

    denominator = min(k_corrected - 1, r_corrected - 1)

    if denominator <= 0:
        return np.nan

    return np.sqrt(phi2_corrected / denominator)


# Use training data only
cat_data = X_train_woe[categorical_features].copy()
cat_data = cat_data.fillna("Missing").astype(str)

# Build association matrix
cramers_matrix = pd.DataFrame(
    np.eye(len(categorical_features)),
    index=categorical_features,
    columns=categorical_features
)

for feature_1, feature_2 in combinations(categorical_features, 2):
    value = cramers_v(
        cat_data[feature_1],
        cat_data[feature_2]
    )

    cramers_matrix.loc[feature_1, feature_2] = value
    cramers_matrix.loc[feature_2, feature_1] = value

cramers_matrix.round(3)

,name_contract_type,name_income_type,name_education_type,name_family_status,name_housing_type,occupation_type,organization_type,flag_own_car,flag_own_realty,region_rating_client_w_city
name_contract_type,1.00,0.06,0.07,0.05,0.03,0.06,0.07,0.00,0.07,0.03
name_income_type,0.06,1.00,0.12,0.13,0.06,0.38,0.56,0.16,0.07,0.13
name_education_type,0.07,0.12,1.00,0.06,0.05,0.22,0.14,0.10,0.03,0.07
name_family_status,0.05,0.13,0.06,1.00,0.07,0.09,0.12,0.17,0.05,0.02
name_housing_type,0.03,0.06,0.05,0.07,1.00,0.04,0.08,0.04,0.23,0.09
occupation_type,0.06,0.38,0.22,0.09,0.04,1.00,0.32,0.26,0.05,0.05
organization_type,0.07,0.56,0.14,0.12,0.08,0.32,1.00,0.20,0.08,0.09
flag_own_car,0.00,0.16,0.10,0.17,0.04,0.26,0.20,1.00,0.00,0.02
flag_own_realty,0.07,0.07,0.03,0.05,0.23,0.05,0.08,0.00,1.00,0.01
region_rating_client_w_city,0.03,0.13,0.07,0.02,0.09,0.05,0.09,0.02,0.01,1.00


The results look good overall. Most categorical predictors have weak associations, so there is no widespread categorical redundancy.

| Predictor pair             | Cramér's V  | Interpretation  |
|-----------------------|-----:|-------:|		
|name_income_type|	0.0559|	Retain|
|organization_type|	0.0726|	Retain|
|occupation_type|	0.0833|	Retain|
|name_education_type|	0.0489|	Retain|
|name_family_status|	0.0233|	Retain|
|name_housing_type|	0.0160|	Retain for now|
|name_contract_type|	0.0149|	Retain for now|
|flag_own_car|	0.0061|	Candidate for removal|
|flag_own_realty|	0.0005|	Candidate for removal|

- income type, organization type or occupation type have meaningful individual IV values and may capture different dimensions of employment and socioeconomic circumstances.
- The two ownership flags are weaker candidates because their individual IV values are very low.

In [57]:
# Examine income type and organization type
pd.crosstab(
    X_train_woe['name_income_type'],
    X_train_woe['organization_type'],
    normalize='index'
).round(3)

organization_type,Agriculture,Bank,Business Entity Type 1,Business Entity Type 2,Business Entity Type 3,Construction,Electricity,Government,Hotel,Housing,...,Self-employed,Services,Trade: type 2,Trade: type 3,Trade: type 7,Transport: type 2,Transport: type 3,Transport: type 4,University,XNA
name_income_type,,,,,,,,,,,,,,,,,,,,,
Commercial associate,0.01,0.02,0.03,0.04,0.35,0.03,0.00,0.02,0.00,0.01,...,0.16,0.01,0.01,0.02,0.04,0.01,0.01,0.02,0.00,0.00
Other,0.00,0.05,0.05,0.00,0.07,0.00,0.03,0.03,0.00,0.00,...,0.20,0.00,0.00,0.00,0.00,0.03,0.00,0.00,0.03,0.42
Pensioner,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00
State servant,0.00,0.00,0.00,0.01,0.03,0.00,0.00,0.14,0.00,0.01,...,0.01,0.00,0.00,0.00,0.00,0.01,0.00,0.01,0.02,0.00
Working,0.01,0.01,0.02,0.05,0.27,0.03,0.00,0.04,0.00,0.01,...,0.17,0.01,0.01,0.01,0.03,0.01,0.01,0.02,0.00,0.00


There are some clear structural relationships:
1. Pensioners: 100% fall under XNA organization type.
2. Other income types: 42% fall under XNA, with another 20% classified as self-employed.
3. State servants: 14% belong to Government, with smaller proportions in other organization types.
4. Working applicants: Concentrated in Business Entity Type 3 (27%) and Self-employed (17%).
5. Commercial associates: Also concentrated in Business Entity Type 3 (35%) and Self-employed (16%).

The association is partly driven by the structural relationship between income categories and organization types, particularly Pensioner and XNA. But the distributions also show variation within the major income categories.

In [ ]:
# Examine occupation type against income type
pd.crosstab(
    X_train_woe['name_income_type'],
    X_train_woe['occupation_type'],
    normalize='index'
).round(3)

occupation_type,Accountants,Cleaning staff,Cooking staff,Core staff,Drivers,HR staff,High skill tech staff,IT staff,Laborers,Low-skill Laborers,Managers,Medicine staff,Missing,Private service staff,Realty agents,Sales staff,Secretaries,Security staff,Waiters/barmen staff
name_income_type,,,,,,,,,,,,,,,,,,,
Commercial associate,0.06,0.01,0.02,0.09,0.07,0.00,0.05,0.00,0.18,0.01,0.12,0.01,0.17,0.01,0.00,0.15,0.01,0.02,0.01
Other,0.05,0.00,0.00,0.07,0.03,0.00,0.00,0.00,0.07,0.05,0.15,0.00,0.55,0.00,0.00,0.00,0.00,0.00,0.03
Pensioner,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00
State servant,0.03,0.01,0.02,0.32,0.04,0.00,0.05,0.00,0.09,0.00,0.10,0.12,0.17,0.00,0.00,0.01,0.01,0.02,0.00
Working,0.03,0.02,0.03,0.09,0.08,0.00,0.04,0.00,0.26,0.01,0.07,0.03,0.16,0.01,0.00,0.13,0.00,0.03,0.01


The cross-tabulation reveals several distinct patterns:

- Pensioners: 100% have missing occupation type. This is expected because occupation may not apply to retired applicants.
- Other income types: 55% have missing occupation type, suggesting that this category combines applicants whose employment status or occupation is not clearly classified.
- State servants: 32% are core staff and 12% are medicine staff, which is consistent with public-sector employment patterns.
- Working applicants: 26% are laborers, 13% sales staff and 9% core staff.
- Commercial associates: More diverse occupational profiles, including laborers (18%), missing (17%), sales staff (15%) and managers (12%).

| Variable            | IV  | Decision |
|-----------------------|-----:|-------:|
|name_income_type	|0.0559	|Retain|
|organization_type	|0.0726	|Retain|
|occupation_type	|0.0833	|Retain|

The reasoning is that they describe different dimensions:
- name_income_type: broad source or classification of income.
- organization_type: sector or type of organization.
- occupation_type: applicant's occupational role.

In [59]:
# Examine car owenership against income type
pd.crosstab(
    X_train_woe['name_income_type'],
    X_train_woe['flag_own_car'],
    normalize='index'
).round(3)

flag_own_car,N,Y
name_income_type,,
Commercial associate,0.61,0.39
Other,0.57,0.42
Pensioner,0.82,0.18
State servant,0.65,0.35
Working,0.63,0.37


Car ownership is not determined by income type. For example, 39% of commercial associates and 37% of working applicants own cars. These categories have different income classifications, yet similar ownership rates. However own_car_age is also highly affected by car ownership and has structural missingness, so retaining both requires careful handling.

In [60]:
# Examine flag_own_realty against name_housing_type
pd.crosstab(
    X_train_woe['flag_own_realty'],
    X_train_woe['name_housing_type'],
    normalize='index'
).round(3)

name_housing_type,Co-op apartment,House / apartment,Municipal apartment,Office apartment,Rented apartment,With parents
flag_own_realty,,,,,,
N,0.00,0.78,0.07,0.01,0.03,0.10
Y,0.00,0.93,0.02,0.01,0.01,0.03


- Applicants who own real estate (Y): 93% live in a house/apartment, compared with 78% among those who do not own real estate.
- Applicants who do not own real estate (N): More are represented in municipal apartments (7%), rented apartments (3%) and living with parents (10%).
- Co-op and office apartments: Almost identical proportions across the two ownership groups.

There is some additional information in the ownership flag, but its standalone predictive contribution is extremely weak.

In [61]:
#Categories to exclude
categorical_exclusions = [
    "flag_own_car",
    "flag_own_realty"
]

## WoE transformation and bin refinement

In [62]:
# Retained Categories
categorical_features = [
    'name_contract_type',
    'name_income_type',
    'name_education_type',
    'name_family_status',
    'name_housing_type',
    'occupation_type',
    'organization_type'
]

# Calculate WoE mappings on the training set
woe_mappings = {}
categorical_iv = {}

for feature in categorical_features:

    woe_table, iv = calculate_woe_iv(
        X_train_woe,
        y_train,
        feature
    )

    woe_mappings[feature] = woe_table['woe'].to_dict()
    categorical_iv[feature] = iv

print("Categorical WoE mappings created successfully.")

Categorical WoE mappings created successfully.


In [63]:
# Transform training and test datasets
X_train_cat_woe = X_train_woe[categorical_features].copy()
X_test_cat_woe = X_test_woe[categorical_features].copy()

for feature in categorical_features:

    X_train_cat_woe[feature] = (
        X_train_cat_woe[feature]
        .fillna("Missing")
        .astype(str)
        .map(woe_mappings[feature])
    )

    X_test_cat_woe[feature] = (
        X_test_cat_woe[feature]
        .fillna("Missing")
        .astype(str)
        .map(woe_mappings[feature])
    )

In [64]:
# Check for unmapped values
unmapped_values = {}

for feature in categorical_features:

    unmapped = X_test_cat_woe[feature].isna().sum()

    unmapped_values[feature] = unmapped

pd.Series(unmapped_values, name="unmapped_test_values")

name_contract_type     0
name_income_type       0
name_education_type    0
name_family_status     0
name_housing_type      0
occupation_type        0
organization_type      0
Name: unmapped_test_values, dtype: int64

In [66]:
# Check that the transformed columns contain numeric values and no missing observations.
print("Training missing values:")
print(X_train_cat_woe.isna().sum())

print("\nTest missing values:")
print(X_test_cat_woe.isna().sum())

print("\nTraining data types:")
print(X_train_cat_woe.dtypes)

Training missing values:
name_contract_type     0
name_income_type       0
name_education_type    0
name_family_status     0
name_housing_type      0
occupation_type        0
organization_type      0
dtype: int64

Test missing values:
name_contract_type     0
name_income_type       0
name_education_type    0
name_family_status     0
name_housing_type      0
occupation_type        0
organization_type      0
dtype: int64

Training data types:
name_contract_type     float64
name_income_type       float64
name_education_type    float64
name_family_status     float64
name_housing_type      float64
occupation_type        float64
organization_type      float64
dtype: object


#### Numerical WoE bin refinement.

In [68]:
# Identify numerical predictors for binning

numerical_features = [
    col for col in X_train.columns
    if col not in (
        categorical_features
        + ordinal_features
        + initial_exclusions
        + ["flag_own_car", "flag_own_realty"]
    )
]

print("Number of numerical predictors:", len(numerical_features))
print(numerical_features)

Number of numerical predictors: 27
['ext_source_1', 'ext_source_2', 'ext_source_3', 'amt_income_total', 'amt_annuity', 'amt_goods_price', 'age_years', 'employment_years', 'years_since_registration', 'years_since_id_change', 'years_since_phone_change', 'own_car_age', 'region_population_relative', 'obs_30_cnt_social_circle', 'def_30_cnt_social_circle', 'amt_req_credit_bureau_year', 'credit_term_months', 'credit_goods_ratio', 'prior_loans_approved', 'prior_recency_days', 'prior_total_credit', 'prior_instalments_paid', 'prior_late_share', 'prior_max_dpd', 'bureau_n_credits', 'bureau_n_active', 'bureau_max_overdue_days']


In [70]:
# Refine the numerical bins

woe_table, iv, bins = numerical_woe_binning(
    X_train,
    y_train,
    feature="ext_source_3",
    q=10
)

print(f"IV: {iv:.4f}")
display(woe_table.round(4))

IV: 0.3296


,applicants,defaults,non_defaults,default_distribution,non_default_distribution,woe,iv_component,default_rate
feature,,,,,,,,
"(-0.000473, 0.228]",19723,"3,928.00","15,795.00",0.20,0.07,-1.04,0.13,0.20
"(0.228, 0.33]",19757,"2,523.00","17,234.00",0.13,0.08,-0.51,0.03,0.13
"(0.33, 0.408]",20001,"1,956.00","18,045.00",0.10,0.08,-0.21,0.00,0.10
"(0.408, 0.476]",19842,"1,576.00","18,266.00",0.08,0.08,0.02,0.00,0.08
"(0.476, 0.535]",19513,"1,206.00","18,307.00",0.06,0.08,0.29,0.01,0.06
"(0.535, 0.59]",19506,"1,042.00","18,464.00",0.05,0.08,0.44,0.01,0.05
"(0.59, 0.643]",20444,990.00,"19,454.00",0.05,0.09,0.55,0.02,0.05
"(0.643, 0.694]",19808,812.00,"18,996.00",0.04,0.08,0.72,0.03,0.04
"(0.694, 0.749]",18902,657.00,"18,245.00",0.03,0.08,0.89,0.04,0.03


Three observations stand out:
- Monotonicity: Default risk consistently declines as the external score increases.
- Risk separation: The lowest-score bin has approximately six times the default rate of the highest-score bin.
- Missing values: The missing group has a 9.3% default rate, above the overall 8.07%. It should remain a separate bin.

In [71]:
numerical_binning_decisions = {
    "ext_source_3": {
        "initial_bins": 10,
        "final_bins": 10,
        "missing_bin": True,
        "monotonic": True,
        "decision": "Retain initial quantile bins"
    }
}

In [72]:
woe_table, iv, bins = numerical_woe_binning(
    X_train,
    y_train,
    feature="ext_source_2",
    q=10
)

print(f"IV: {iv:.4f}")
display(woe_table.round(4))

IV: 0.3030


,applicants,defaults,non_defaults,default_distribution,non_default_distribution,woe,iv_component,default_rate
feature,,,,,,,,
"(-0.0009999183, 0.215]",24548,"4,487.00","20,061.00",0.23,0.09,-0.93,0.13,0.18
"(0.215, 0.34]",24548,"2,970.00","21,578.00",0.15,0.10,-0.45,0.02,0.12
"(0.34, 0.44]",24547,"2,423.00","22,124.00",0.12,0.10,-0.22,0.01,0.10
"(0.44, 0.512]",24549,"2,065.00","22,484.00",0.10,0.10,-0.04,0.00,0.08
"(0.512, 0.566]",24551,"1,833.00","22,718.00",0.09,0.10,0.08,0.00,0.07
"(0.566, 0.608]",24543,"1,611.00","22,932.00",0.08,0.10,0.22,0.00,0.07
"(0.608, 0.646]",24548,"1,437.00","23,111.00",0.07,0.10,0.35,0.01,0.06
"(0.646, 0.682]",24550,"1,234.00","23,316.00",0.06,0.10,0.51,0.02,0.05
"(0.682, 0.722]",24547,"1,033.00","23,514.00",0.05,0.10,0.69,0.04,0.04


In [73]:
numerical_binning_decisions["ext_source_2"] = {
    "initial_bins": 10,
    "final_bins": 10,
    "missing_bin": False,
    "monotonic": True,
    "decision": "Retain initial quantile bins; no separate missing bin"
}

In [74]:
woe_table, iv, bins = numerical_woe_binning(
    X_train,
    y_train,
    feature="ext_source_1",
    q=10
)

print(f"IV: {iv:.4f}")
display(woe_table.round(4))

IV: 0.1512


,applicants,defaults,non_defaults,default_distribution,non_default_distribution,woe,iv_component,default_rate
feature,,,,,,,,
"(0.013600000000000001, 0.212]",10742,"1,870.00","8,872.00",0.09,0.04,-0.88,0.05,0.17
"(0.212, 0.296]",10741,"1,250.00","9,491.00",0.06,0.04,-0.41,0.01,0.12
"(0.296, 0.368]",10741,981.00,"9,760.00",0.05,0.04,-0.14,0.00,0.09
"(0.368, 0.437]",10741,907.00,"9,834.00",0.05,0.04,-0.05,0.00,0.08
"(0.437, 0.505]",10742,732.00,"10,010.00",0.04,0.04,0.18,0.00,0.07
"(0.505, 0.573]",10741,640.00,"10,101.00",0.03,0.04,0.33,0.00,0.06
"(0.573, 0.64]",10741,555.00,"10,186.00",0.03,0.04,0.48,0.01,0.05
"(0.64, 0.71]",10741,466.00,"10,275.00",0.02,0.05,0.66,0.01,0.04
"(0.71, 0.787]",10741,367.00,"10,374.00",0.02,0.05,0.91,0.02,0.03


In [75]:
numerical_binning_decisions["ext_source_1"] = {
    "initial_bins": 10,
    "final_bins": 10,
    "missing_bin": True,
    "monotonic": True,
    "decision": "Retain initial quantile bins and separate missing bin"
}

In [76]:
# Examine credit_term_months
woe_table, iv, bins = numerical_woe_binning(
    X_train,
    y_train,
    feature="credit_term_months",
    q=10
)

print(f"IV: {iv:.4f}")
display(woe_table.round(4))

IV: 0.1419


,applicants,defaults,non_defaults,default_distribution,non_default_distribution,woe,iv_component,default_rate
feature,,,,,,,,
"(8.036000000000001, 10.528]",24613,"1,212.00","23,401.00",0.06,0.10,0.53,0.02,0.05
"(10.528, 14.658]",24625,"2,803.00","21,822.00",0.14,0.10,-0.38,0.02,0.11
"(14.658, 17.791]",24564,"2,411.00","22,153.00",0.12,0.10,-0.21,0.01,0.10
"(17.791, 19.537]",24674,"2,483.00","22,191.00",0.12,0.10,-0.24,0.01,0.10
"(19.537, 20.0]",26989,"1,584.00","25,405.00",0.08,0.11,0.34,0.01,0.06
"(20.0, 22.038]",22134,"3,136.00","18,998.00",0.16,0.08,-0.63,0.05,0.14
"(22.038, 25.215]",24661,"1,266.00","23,395.00",0.06,0.10,0.48,0.02,0.05
"(25.215, 30.143]",24685,"1,817.00","22,868.00",0.09,0.10,0.10,0.00,0.07
"(30.143, 34.063]",25155,"1,771.00","23,384.00",0.09,0.10,0.15,0.00,0.07


For credit_term_months, the initial 10-quantile binning is clearly non-monotonic. The default rate jumps around rather than following a consistent risk pattern. The pattern suggests that repayment term may have a nonlinear relationship with default risk

In [77]:
X_train["credit_term_months"].describe()

count   245,998.00
mean         21.62
std           7.82
min           8.04
25%          15.65
50%          20.00
75%          27.10
max          45.31
Name: credit_term_months, dtype: float64

The repayment terms are concentrated around 16–27 months, but extend across a fairly wide range. The initial quantile bins also show that risk is not consistently increasing or decreasing with term length.

In [79]:
term_bins = [0, 12, 18, 24, 36, np.inf]
term_labels = [
    "Up to 12 months",
    "13–18 months",
    "19–24 months",
    "25–36 months",
    "Above 36 months"
]

term_groups = pd.cut(
    X_train["credit_term_months"],
    bins=term_bins,
    labels=term_labels,
    include_lowest=True
)

term_groups = term_groups.astype("object")

term_table = calculate_woe_iv(
    pd.DataFrame({"credit_term_group": term_groups}),
    y_train,
    "credit_term_group"
)

term_table[0], term_table[1]

(                 applicants  defaults  non_defaults  default_distribution  \
 feature                                                                     
 13–18 months          48432  4,967.00     43,465.00                  0.25   
 19–24 months          88572  8,068.00     80,504.00                  0.41   
 25–36 months          70412  4,800.00     65,612.00                  0.24   
 Up to 12 months       28436  1,562.00     26,874.00                  0.08   
 Above 36 months       10146    463.00      9,683.00                  0.02   
 Missing                  10      0.00         10.00                  0.00   
 
                  non_default_distribution   woe  iv_component  default_rate  
 feature                                                                      
 13–18 months                         0.19 -0.26          0.02          0.10  
 19–24 months                         0.36 -0.13          0.01          0.09  
 25–36 months                         0.29  0.18          

- Default risk is highest among applicants with terms of 13–18 months.

- Risk declines across the longer repayment-term groups.

- The shortest and longest terms have the lowest observed default rates.

- The relationship is not simply linear: medium-short terms carry more risk than both shorter and longer terms.

In [80]:
numerical_binning_decisions["credit_term_months"] = {
    "initial_bins": 10,
    "final_bins": 5,
    "missing_bin": False,
    "monotonic": True,
    "decision": "Use business-readable repayment-term groups"
}

In [81]:
# Examine Employment Years
woe_table, iv, bins = numerical_woe_binning(
    X_train,
    y_train,
    feature="employment_years",
    q=10
)

print(f"IV: {iv:.4f}")
display(woe_table.round(4))

IV: 0.1121


,applicants,defaults,non_defaults,default_distribution,non_default_distribution,woe,iv_component,default_rate
feature,,,,,,,,
"(-0.001, 0.9]",21104,"2,341.00","18,763.00",0.12,0.08,-0.35,0.01,0.11
"(0.9, 1.7]",20786,"2,400.00","18,386.00",0.12,0.08,-0.40,0.02,0.12
"(1.7, 2.5]",19637,"2,159.00","17,478.00",0.11,0.08,-0.34,0.01,0.11
"(2.5, 3.4]",19843,"2,036.00","17,807.00",0.10,0.08,-0.26,0.01,0.10
"(3.4, 4.5]",20281,"1,908.00","18,373.00",0.10,0.08,-0.17,0.00,0.09
"(4.5, 5.9]",19961,"1,667.00","18,294.00",0.08,0.08,-0.04,0.00,0.08
"(5.9, 7.6]",19752,"1,497.00","18,255.00",0.08,0.08,0.07,0.00,0.08
"(7.6, 10.1]",20678,"1,437.00","19,241.00",0.07,0.09,0.16,0.00,0.07
"(10.1, 14.6]",19797,"1,151.00","18,646.00",0.06,0.08,0.35,0.01,0.06


- Default rates decline steadily from 11.1% in the shortest employment-duration bin to 4.5% in the longest.

- WoE increases consistently from -0.35 to +0.63.

- This suggests applicants with longer employment histories tend to have lower observed default rates.

- The missing group contains 44,143 applicants (approximately 18% of the training sample), with a 5.4% default rate and WoE of +0.44.

In [82]:
numerical_binning_decisions["employment_years"] = {
    "initial_bins": 10,
    "final_bins": 10,
    "missing_bin": True,
    "monotonic": True,
    "decision": "Retain initial quantile bins and separate missing bin"
}

In [83]:
# Examine amt_goods_price
woe_table, iv, bins = numerical_woe_binning(
    X_train,
    y_train,
    feature="amt_goods_price",
    q=10
)

print(f"IV: {iv:.4f}")
display(woe_table.round(4))

IV: 0.0913


,applicants,defaults,non_defaults,default_distribution,non_default_distribution,woe,iv_component,default_rate
feature,,,,,,,,
"(40499.999, 180000.0]",31580,"2,322.00","29,258.00",0.12,0.13,0.10,0.00,0.07
"(180000.0, 225000.0]",25564,"2,227.00","23,337.00",0.11,0.10,-0.08,0.00,0.09
"(225000.0, 270000.0]",22429,"1,763.00","20,666.00",0.09,0.09,0.03,0.00,0.08
"(270000.0, 378000.0]",18895,"1,927.00","16,968.00",0.10,0.07,-0.26,0.01,0.10
"(378000.0, 450000.0]",27768,"3,687.00","24,081.00",0.19,0.11,-0.56,0.04,0.13
"(450000.0, 526500.0]",21454,"1,412.00","20,042.00",0.07,0.09,0.22,0.00,0.07
"(526500.0, 675000.0]",33740,"2,853.00","30,887.00",0.14,0.14,-0.05,0.00,0.08
"(675000.0, 819000.0]",15240,850.00,"14,390.00",0.04,0.06,0.40,0.01,0.06
"(819000.0, 1098000.0]",24550,"1,586.00","22,964.00",0.08,0.10,0.24,0.01,0.06


The main pattern is:
- The first bin (approximately 40,500–180,000) has a 7.4% default rate.
- Risk rises through the middle bins, reaching approximately 13.3% for goods prices between 378,000 and 450,000.
- It then falls to approximately 5.0% in the highest bin.
- The missing group has only 221 applicants and an 7.7% default rate.

This suggests a nonlinear relationship. Higher goods prices do not consistently correspond to higher or lower default risk.

In [84]:
# Grouping based on observed risk pattern
goods_bins = [0, 225000, 378000, 450000, 675000,
              819000, np.inf]

goods_labels = [
    "Up to 225K",
    "225K–378K",
    "378K–450K",
    "450K–675K",
    "675K–819K",
    "Above 819K"
]

goods_groups = pd.cut(
    X_train["amt_goods_price"],
    bins=goods_bins,
    labels=goods_labels,
    include_lowest=True
).astype("object")

goods_table = calculate_woe_iv(
    pd.DataFrame({"goods_price_group": goods_groups}),
    y_train,
    "goods_price_group"
)

goods_table[0], goods_table[1]

(            applicants  defaults  non_defaults  default_distribution  \
 feature                                                                
 378K–450K        27768  3,687.00     24,081.00                  0.19   
 225K–378K        41324  3,690.00     37,634.00                  0.19   
 Up to 225K       57144  4,549.00     52,595.00                  0.23   
 Missing            221     17.00        204.00                  0.00   
 450K–675K        55194  4,265.00     50,929.00                  0.21   
 Above 819K       49117  2,802.00     46,315.00                  0.14   
 675K–819K        15240    850.00     14,390.00                  0.04   
 
             non_default_distribution   woe  iv_component  default_rate  
 feature                                                                 
 378K–450K                       0.11 -0.56          0.04          0.13  
 225K–378K                       0.17 -0.11          0.00          0.09  
 Up to 225K                      0.23  0.02  

I retain these six groups provisionally, with two qualifications:

1. The relationship is not fully monotonic, so don't force it into a monotonic pattern.
2. amt_goods_price is highly correlated with amt_credit (Spearman 0.98), and its VIF is 116.81. We should eventually compare models with and without it rather than assume it adds independent predictive value.

The 221 missing observations can be assigned using a training-derived imputation rule rather than a separate bin.

In [85]:
numerical_binning_decisions["amt_goods_price"] = {
    "initial_bins": 10,
    "final_bins": 6,
    "missing_bin": False,
    "monotonic": False,
    "decision": "Retain six exploratory groups; assess redundancy"
}

In [87]:
# Examine age_years
woe_table, iv, bins = numerical_woe_binning(
    X_train,
    y_train,
    feature="age_years",
    q=10
)

print(f"IV: {iv:.4f}")
display(woe_table.round(4))

IV: 0.0868


,applicants,defaults,non_defaults,default_distribution,non_default_distribution,woe,iv_component,default_rate
feature,,,,,,,,
"(20.499, 28.1]",24679,"2,876.00","21,803.00",0.14,0.10,-0.41,0.02,0.12
"(28.1, 32.0]",24884,"2,739.00","22,145.00",0.14,0.10,-0.34,0.01,0.11
"(32.0, 36.0]",24757,"2,427.00","22,330.00",0.12,0.10,-0.21,0.01,0.10
"(36.0, 39.5]",24765,"2,187.00","22,578.00",0.11,0.10,-0.10,0.00,0.09
"(39.5, 43.1]",24208,"1,942.00","22,266.00",0.10,0.10,0.01,0.00,0.08
"(43.1, 47.1]",24411,"1,883.00","22,528.00",0.09,0.10,0.05,0.00,0.08
"(47.1, 51.7]",24873,"1,756.00","23,117.00",0.09,0.10,0.14,0.00,0.07
"(51.7, 56.0]",24403,"1,535.00","22,868.00",0.08,0.10,0.27,0.01,0.06
"(56.0, 60.7]",24632,"1,326.00","23,306.00",0.07,0.10,0.43,0.02,0.05


age_years has a strong, consistent relationship with default risk. Its initial 10 quantile bins show an almost perfectly monotonic pattern.

In [88]:
numerical_binning_decisions["age_years"] = {
    "initial_bins": 10,
    "final_bins": 10,
    "missing_bin": False,
    "monotonic": True,
    "decision": "Retain initial quantile bins"
}

In [89]:
# Automating the rest of the screening
inspected_features = [
    "ext_source_1",
    "ext_source_2",
    "ext_source_3",
    "credit_term_months",
    "employment_years",
    "amt_goods_price",
    "age_years"
]

remaining_features = [
    col for col in numerical_features
    if col not in inspected_features
]

diagnostic_results = []

for feature in remaining_features:

    woe_table, iv, bins = numerical_woe_binning(
        X_train, y_train, feature, q=10
    )

    # Exclude missing values when assessing monotonicity
    regular_bins = [
        idx for idx in woe_table.index
        if isinstance(idx, pd.Interval)
    ]

    regular_bins = sorted(regular_bins, key=lambda x: x.left)

    rates = woe_table.loc[regular_bins, "default_rate"].values

    increasing = np.all(np.diff(rates) >= 0)
    decreasing = np.all(np.diff(rates) <= 0)

    monotonic = increasing or decreasing

    # Missing-value information
    missing_count = X_train[feature].isna().sum()
    missing_pct = X_train[feature].isna().mean() * 100

    bin_counts = woe_table.loc[
        regular_bins, "applicants"
    ]

    diagnostic_results.append({
        "variable": feature,
        "IV": iv,
        "number_of_bins": len(regular_bins),
        "monotonic": monotonic,
        "min_bin_applicants": bin_counts.min(),
        "missing_count": missing_count,
        "missing_pct": missing_pct
    })

numerical_diagnostics = pd.DataFrame(diagnostic_results)

numerical_diagnostics = numerical_diagnostics.sort_values(
    "IV", ascending=False
).reset_index(drop=True)

numerical_diagnostics.round(4)

,variable,IV,number_of_bins,monotonic,min_bin_applicants,missing_count,missing_pct
0,credit_goods_ratio,0.07,8,False,11632,221,0.09
1,prior_late_share,0.06,6,True,22848,12691,5.16
2,bureau_n_active,0.05,5,True,17645,35244,14.33
3,years_since_phone_change,0.04,9,False,24343,1,0.00
4,prior_max_dpd,0.04,10,False,12520,12691,5.16
5,years_since_id_change,0.04,10,True,24439,0,0.00
6,region_population_relative,0.03,10,False,17488,0,0.00
7,prior_total_credit,0.03,10,False,23200,13985,5.68
8,years_since_registration,0.03,10,False,24559,0,0.00
9,amt_annuity,0.02,10,False,24458,10,0.00


In [90]:
# Combine exact IV values with the automated diagnostics
candidate_selection = (
    all_iv_summary[["variable", "IV"]]
    .merge(
        numerical_diagnostics.drop(columns=["IV"]),
        on="variable",
        how="left"
    )
)

# Add a preliminary screening category
def classify_predictor(iv):
    if iv >= 0.10:
        return "Strong"
    elif iv >= 0.02:
        return "Moderate"
    else:
        return "Weak"

candidate_selection["IV_category"] = (
    candidate_selection["IV"].apply(classify_predictor)
)

candidate_selection = candidate_selection.sort_values(
    "IV", ascending=False
).reset_index(drop=True)

candidate_selection.round(4)

,variable,IV,number_of_bins,monotonic,min_bin_applicants,missing_count,missing_pct,IV_category
0,ext_source_3,0.33,NaN,NaN,NaN,NaN,NaN,Strong
1,ext_source_2,0.30,NaN,NaN,NaN,NaN,NaN,Strong
2,ext_source_1,0.15,NaN,NaN,NaN,NaN,NaN,Strong
3,credit_term_months,0.14,NaN,NaN,NaN,NaN,NaN,Strong
4,employment_years,0.11,NaN,NaN,NaN,NaN,NaN,Strong
5,amt_goods_price,0.09,NaN,NaN,NaN,NaN,NaN,Moderate
6,age_years,0.09,NaN,NaN,NaN,NaN,NaN,Moderate
7,occupation_type,0.08,NaN,NaN,NaN,NaN,NaN,Moderate
8,organization_type,0.07,NaN,NaN,NaN,NaN,NaN,Moderate
9,credit_goods_ratio,0.07,8.00,False,"11,632.00",221.00,0.09,Moderate


1. The seven manually inspected predictors have NaN diagnostic fields because they were not included in numerical_diagnostics. Their IV values are still available.
2. The table includes predictors we had already provisionally excluded because of multicollinearity or weak contribution. We should not automatically bring them back into the model.

Provisional exclusions
- flag_own_car
- flag_own_realty
- own_car_age — 66% missing and structurally related to car ownership.
- amt_credit — highly redundant with amt_goods_price.
- credit_income_ratio and annuity_income_ratio — high VIF and overlapping information.
- cnt_children and cnt_fam_members — weak IV and correlated.
- bureau_max_overdue_days — only one non-missing bin in the initial quantile analysis.

In [92]:

# The preliminary feature list
preliminary_features = [
    # External credit scores
    "ext_source_1",
    "ext_source_2",
    "ext_source_3",

    # Applicant and loan characteristics
    "age_years",
    "employment_years",
    "amt_goods_price",
    "amt_annuity",
    "credit_term_months",
    "credit_goods_ratio",
    "region_population_relative",
    "region_rating_client_w_city",
    "years_since_registration",
    "years_since_id_change",
    "years_since_phone_change",

    # Previous loan behaviour
    "prior_late_share",
    "prior_max_dpd",
    "prior_loans_approved",
    "prior_total_credit",
    "prior_instalments_paid",
    "prior_recency_days",

    # Credit bureau
    "bureau_n_active",
    "bureau_n_credits",

    # Credit enquiries and social circle
    "amt_req_credit_bureau_year",
    "obs_30_cnt_social_circle",
    "def_30_cnt_social_circle",

    # Categorical predictors
    "name_contract_type",
    "name_income_type",
    "name_education_type",
    "name_family_status",
    "name_housing_type",
    "occupation_type",
    "organization_type"
]

In [94]:
# Verify preliminary feature list
missing_features = [
    col for col in preliminary_features
    if col not in X_train.columns or col not in X_test.columns
]

print("Preliminary predictors:", len(preliminary_features))
print("Training feature shape:", X_train[preliminary_features].shape)
print("Testing feature shape:", X_test[preliminary_features].shape)
print("Missing predictors:", missing_features)

assert len(preliminary_features) == 32
assert len(set(preliminary_features)) == 32
assert len(missing_features) == 0
assert "target" not in preliminary_features
assert "sk_id_curr" not in preliminary_features

print("Verified: 32 unique preliminary predictors available in both datasets.")

Preliminary predictors: 32
Training feature shape: (246008, 32)
Testing feature shape: (61503, 32)
Missing predictors: []
Verified: 32 unique preliminary predictors available in both datasets.
